In [ ]:
!aws configure set aws_access_key_id #hided for security
!aws configure set aws_secret_access_key #hided for security
!aws configure set default.region us-east-1

In [2]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-3-87-202-243.compute-1.amazonaws.com:5000")

In [3]:
import pandas as pd

df = pd.read_csv('youtube_preprocessing.csv').dropna()
df.shape

(3118, 3)

In [4]:
!pip install sentence-transformers mlflow

In [5]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)

c:\Users\Urmi Kanrar\anaconda3\envs\sentiment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
mlflow.set_experiment("MPNet_SVM_Experiment")
mlflow.sklearn.autolog()

2026/03/02 18:28:26 INFO mlflow.tracking.fluent: Experiment with name 'MPNet_SVM_Experiment' does not exist. Creating a new experiment.
2026/03/02 18:28:28 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.5.2, but the installed version is 1.8.0. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.


In [8]:
with mlflow.start_run(run_name="MPNet_SVM_balanced"):

    # -------------------------
    # Load Stronger Embedding Model
    # -------------------------
    embedder = SentenceTransformer("all-mpnet-base-v2")
    mlflow.log_param("embedding_model", "all-mpnet-base-v2")

    sentences = df['Comment'].tolist()
    y = df['sentiment_encoded'].values

    # -------------------------
    # Generate embeddings
    # -------------------------
    embeddings = embedder.encode(
        sentences,
        batch_size=32,
        show_progress_bar=True
    )

    # -------------------------
    # Train-test split
    # -------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        embeddings,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    # -------------------------
    # Train SVM (balanced)
    # -------------------------
    svm_model = LinearSVC(
        C=2.0,
        max_iter=5000,
        class_weight='balanced'
    )

    svm_model.fit(X_train, y_train)

    # -------------------------
    # Predictions
    # -------------------------
    y_pred = svm_model.predict(X_test)

    # -------------------------
    # Metrics
    # -------------------------
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')
    macro_precision = precision_score(y_test, y_pred, average='macro')
    macro_recall = recall_score(y_test, y_pred, average='macro')

    # -------------------------
    # Log metrics
    # -------------------------
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("macro_f1", macro_f1)
    mlflow.log_metric("weighted_f1", weighted_f1)
    mlflow.log_metric("macro_precision", macro_precision)
    mlflow.log_metric("macro_recall", macro_recall)

    print(classification_report(y_test, y_pred))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 699.07it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 98/98 [01:16<00:00,  1.29it/s]


              precision    recall  f1-score   support

          -1       0.58      0.62      0.60       209
           0       0.46      0.51      0.48        99
           1       0.72      0.66      0.69       316

    accuracy                           0.62       624
   macro avg       0.59      0.60      0.59       624
weighted avg       0.63      0.62      0.63       624



2026/03/02 18:49:16 INFO mlflow.tracking._tracking_service.client: 🏃 View run MPNet_SVM_balanced at: http://ec2-3-87-202-243.compute-1.amazonaws.com:5000/#/experiments/16/runs/a75418385b5c4c3d9c4eb97b6a638a36.
2026/03/02 18:49:16 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://ec2-3-87-202-243.compute-1.amazonaws.com:5000/#/experiments/16.
